# 04 - Final Recall Test (3-slice v2)

Évaluation finale avec ResNet-50 ImageNet + crop_margin=30 + 3-slice. Écrit `data/results/dice_final_report_msd_recall_3slice_v2.csv`.

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'clean-structure',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run — data/ et work_dirs/ doivent déjà être peuplés.')

→ detect GPU
✓ deps already installed, skipping
✓  /content/INF8225_Projet/data already linked
✓  /content/INF8225_Projet/work_dirs already linked
✓  /content/INF8225_Projet/MedSAM/work_dir already linked
✓  /content/INF8225_Projet/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth already linked

Project : /content/INF8225_Projet
Drive   : /content/drive/MyDrive/Projet_Medsam
Device  : NVIDIA A100-SXM4-40GB
Torch   : 2.4.0+cu121

⚠  numpy was already loaded in this kernel before setup reinstalled it.
   Runtime → Restart session, then run your imports again
   (no need to rerun setup — deps are pinned on disk).
cwd: /content/INF8225_Projet


In [ ]:
#@title Verify 3-slice artifacts and MedSAM
from pathlib import Path
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
required = [
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("msd_implementation/configs/grounding_dino/pancreas_tumor.py"),
    Path("MedSAM/work_dir/MedSAM/medsam_vit_b.pth"),
    Path("outputs/msd_implementation/three_slice_context/metrics/optimal_threshold_three_slice_context.txt"),
]
required += [checkpoint_dir / f"three_slice_fold_{i}.pth" for i in range(1, 6)]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}. Lance d'abord les notebooks 01, 02, 03."

OK      data/MSD_pancreas/test.json
OK      work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
OK      work_dirs/tumor_config_v3/tumor_config_v3.py
OK      MedSAM/work_dir/MedSAM/medsam_vit_b.pth
OK      optimal_threshold_3slice_v2.txt
OK      /content/drive/MyDrive/Projet_Medsam/resnet50_3slice_fold_1.pth
OK      /content/drive/MyDrive/Projet_Medsam/resnet50_3slice_fold_2.pth
OK      /content/drive/MyDrive/Projet_Medsam/resnet50_3slice_fold_3.pth
OK      /content/drive/MyDrive/Projet_Medsam/resnet50_3slice_fold_4.pth
OK      /content/drive/MyDrive/Projet_Medsam/resnet50_3slice_fold_5.pth


In [ ]:
import os
os.environ["RESNET_THRESHOLD_OVERRIDE"] = "0.10"

In [ ]:
#@title Run pipeline step (3-slice test)
from collections import deque
import subprocess
import sys

cmd = [sys.executable, "-u", "-m", "msd_implementation.pipelines.three_slice_context.evaluate"]
print("Running:", " ".join(cmd))

last_lines = deque(maxlen=160)
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    last_lines.append(line)

return_code = proc.wait()
if return_code != 0:
    print("\n" + "=" * 80)
    print(f"test_recall_3slice failed with exit code {return_code}. Last captured lines:")
    print("=" * 80)
    print("".join(last_lines))
    raise RuntimeError(f"test_recall_3slice failed with exit code {return_code}")

Running: /usr/bin/python3 -u -m experiments.three_slice_v2.test_v2
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
Appareil detecte : cuda
Seuil 3-slice v2 charge depuis optimal_threshold_3slice_v2.json : 0.25
Override RESNET_THRESHOLD_OVERRIDE : 0.10
Loads checkpoint by local backend from path: work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY

In [ ]:
#@title Inspect final 3-slice report
from pathlib import Path
import pandas as pd
report = Path("outputs/msd_implementation/three_slice_context/metrics/dice_final_report_three_slice_context.csv")
print(("OK     " if report.exists() else "MISSING"), report)
assert report.exists(), "Final 3-slice report CSV missing"
df = pd.read_csv(report)
display(df.head())
print("rows:", len(df))
print("mean dice:", df["final_dice"].mean())

OK      data/results/dice_final_report_msd_recall_3slice_v2.csv


,file_name,has_tumor,final_dice,n_candidates,n_selected,best_candidate_score,best_resnet_prob,best_dino_score,selected_scores,fn_cause
0,test/images/pancreas_015_s25.png,False,1.000000,1,0,0.041634,0.044974,0.231877,[],NaN
1,test/images/pancreas_015_s30.png,True,0.645598,2,2,0.652469,0.737959,0.214152,"[0.6524694934487343, 0.49643103778362274]",NaN
2,test/images/pancreas_015_s32.png,True,0.746345,2,2,0.974882,0.978873,0.225008,"[0.974882232490927, 0.7224997952580452]",NaN
3,test/images/pancreas_015_s34.png,True,0.490323,1,1,0.973573,0.978958,0.271210,[0.973573375493288],NaN
4,test/images/pancreas_019_s34.png,True,0.511765,3,2,0.980145,0.982240,0.177389,"[0.9801452960819006, 0.9664446082897484]",NaN


rows: 100
mean dice: 0.6628979771011533
